# Online Retail II — Data Profiling & Quality Assessment

Source: https://archive.ics.uci.edu/dataset/502/online%2Bretail

This notebook performs exploratory data profiling and data quality assessment on the combined UCI Online Retail II dataset using Python and pandas.

The objective is to understand the structure, completeness, consistency, and validity of the data before performing cleaning, transformation, or business analysis. The profiling process evaluates dataset dimensions, data types, missing values, distinct values, duplicate records, numerical distributions, date coverage, and potentially invalid or unusual values.

Findings from this notebook will be used to identify data quality issues, document cleaning decisions, and determine whether the dataset is suitable for downstream revenue analysis and reporting.


#### Project Setup

In [10]:
# Install packages
from google.colab import drive
import pandas as pd

# Connect to Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
# Save file path of online retail data
file_path = "/content/drive/MyDrive/Working/Resume 📃/Portfolio/Retail Revenue Intelligence/Data/online_retail_raw.xlsx"

# Load all sheets
sheets = pd.read_excel(file_path, sheet_name=None)

# Combine them into one DataFrame
df = pd.concat(sheets.values(), ignore_index=True)

# Add a column to keep track of which tab the data came from
df = pd.concat(
    [sheet.assign(Source_Sheet=name) for name, sheet in sheets.items()],
    ignore_index=True
)

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Source_Sheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010


### Dataset Overview

The two source worksheets were combined into a single DataFrame for profiling.
This section reviews the size, structure, data types, and sample records of the
combined dataset before performing detailed data-quality checks.

In [12]:
# Dataset dimensions
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

# Column names and data types
df.info()

Rows: 1,067,371
Columns: 9
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   Invoice       1067371 non-null  object        
 1   StockCode     1067371 non-null  object        
 2   Description   1062989 non-null  object        
 3   Quantity      1067371 non-null  int64         
 4   InvoiceDate   1067371 non-null  datetime64[ns]
 5   Price         1067371 non-null  float64       
 6   Customer ID   824364 non-null   float64       
 7   Country       1067371 non-null  object        
 8   Source_Sheet  1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 73.3+ MB


### Field-Level Data Profiling

This section creates a field-level profile of the combined dataset to assess its structure and completeness. Each column is evaluated using the following metrics:

* **Field** — Name of the column in the dataset.
* **Data Type** — pandas data type assigned to the field, such as `object`, `int64`, `float64`, or `datetime64`.
* **Total Rows** — Total number of records in the combined dataset.
* **Missing Count** — Number of null or missing values in each field.
* **Missing %** — Percentage of records containing a missing value for each field.
* **Distinct Count** — Number of unique non-null values present in each field.

These metrics provide an initial view of data completeness and cardinality and help identify fields that may require additional validation or cleaning before analysis.


In [13]:
profile = pd.DataFrame({
    "Field": df.columns,
    "Data Type": df.dtypes,
    "Total Rows": len(df),
    "Missing Count": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(2),
    "Distinct Count": df.nunique()
})

profile

,Field,Data Type,Total Rows,Missing Count,Missing %,Distinct Count
Invoice,Invoice,object,1067371,0,0.00,53628
StockCode,StockCode,object,1067371,0,0.00,5305
Description,Description,object,1067371,4382,0.41,5698
Quantity,Quantity,int64,1067371,0,0.00,1057
InvoiceDate,InvoiceDate,datetime64[ns],1067371,0,0.00,47635
Price,Price,float64,1067371,0,0.00,2807
Customer ID,Customer ID,float64,1067371,243007,22.77,5942
Country,Country,object,1067371,0,0.00,43
Source_Sheet,Source_Sheet,object,1067371,0,0.00,2


### Duplicate Record Analysis

This section evaluates the dataset for duplicate records that could affect transaction counts, quantities, revenue calculations, and other downstream analyses.

The analysis begins by identifying **exact duplicate rows**, where all fields contain identical values. Duplicate records will be investigated before any records are removed because identical transaction lines may represent either true data duplication or legitimate repeated purchases within the same invoice.


In [14]:
# Creates a new dataset containing all rows that have an exact match elsewhere in the data
duplicates = df[df.duplicated(keep=False)]
duplicates.shape

(23430, 9)

In [15]:
duplicates.sort_values(
    by=["Invoice", "StockCode", "InvoiceDate"]
).head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Source_Sheet
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,Year 2009-2010
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,Year 2009-2010
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom,Year 2009-2010


In [16]:
duplicate_pct = (df.duplicated().mean() * 100).round(2)

print(f"Exact duplicate rows: {df.duplicated().sum():,}")
print(f"Duplicate percentage: {duplicate_pct}%")

Exact duplicate rows: 12,133
Duplicate percentage: 1.14%


### Value Validation

This section evaluates key transactional fields for values that may represent returns, cancellations, adjustments, data-entry issues, or other conditions requiring special treatment before revenue analysis.

####Quantity Validation

Quantity represents the number of units associated with each transaction line. Because quantity directly affects revenue calculations, zero and negative values are examined separately from normal positive sales quantities.

In [17]:
df["Quantity"].describe()

,Quantity
count,1.067371e+06
mean,9.938898e+00
std,1.727058e+02
min,-8.099500e+04
25%,1.000000e+00
50%,3.000000e+00
75%,1.000000e+01
max,8.099500e+04


In [18]:
print(f"Negative quantities: {(df['Quantity'] < 0).sum():,}")
print(f"Zero quantities: {(df['Quantity'] == 0).sum():,}")
print(f"Positive quantities: {(df['Quantity'] > 0).sum():,}")

Negative quantities: 22,950
Zero quantities: 0
Positive quantities: 1,044,421


In [19]:
negative_qty = df[df["Quantity"] < 0] # Creates a dataset containing only transactions with negative quantities

negative_qty.head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Source_Sheet
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,Year 2009-2010
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia,Year 2009-2010
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia,Year 2009-2010
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia,Year 2009-2010
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,Year 2009-2010
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.0,Australia,Year 2009-2010
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321.0,Australia,Year 2009-2010
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321.0,Australia,Year 2009-2010
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,Year 2009-2010
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom,Year 2009-2010


In [20]:
negative_qty["Quantity"].describe()

,Quantity
count,22950.000000
mean,-46.365054
std,788.250496
min,-80995.000000
25%,-11.000000
50%,-2.000000
75%,-1.000000
max,-1.000000


In [21]:
negative_qty["Invoice"].head(20)

,Invoice
178,C489449
179,C489449
180,C489449
181,C489449
182,C489449
183,C489449
184,C489449
185,C489449
186,C489449
196,C489459


In [22]:
negative_qty["Invoice"].astype(str).str.startswith("C").value_counts()

,count
Invoice,
True,19493
False,3457


In [23]:
negative_qty_no_c = negative_qty[
    ~negative_qty["Invoice"].astype(str).str.startswith("C")
]

negative_qty_no_c.head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Source_Sheet
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom,Year 2009-2010
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom,Year 2009-2010
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom,Year 2009-2010
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom,Year 2009-2010
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom,Year 2009-2010
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom,Year 2009-2010
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom,Year 2009-2010
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom,Year 2009-2010
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0,NaN,United Kingdom,Year 2009-2010
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom,Year 2009-2010


In [24]:
print(f"Total non-C negative quantity records: {len(negative_qty_no_c):,}")
print(f"Price = 0: {(negative_qty_no_c['Price'] == 0).sum():,}")
print(f"Missing Customer ID: {negative_qty_no_c['Customer ID'].isna().sum():,}")

Total non-C negative quantity records: 3,457
Price = 0: 3,457
Missing Customer ID: 3,457


#### Price Validation

Price represents the unit price associated with each transaction line. Because price directly affects revenue calculations, zero and negative prices are examined separately from normal positive prices to identify potential adjustments, non-sale transactions, or other values requiring further investigation.

In [25]:
df["Price"].describe()

,Price
count,1.067371e+06
mean,4.649388e+00
std,1.235531e+02
min,-5.359436e+04
25%,1.250000e+00
50%,2.100000e+00
75%,4.150000e+00
max,3.897000e+04


In [26]:
print(f"Negative prices: {(df['Price'] < 0).sum():,}")
print(f"Zero prices: {(df['Price'] == 0).sum():,}")
print(f"Positive prices: {(df['Price'] > 0).sum():,}")

Negative prices: 5
Zero prices: 6,202
Positive prices: 1,061,164


In [27]:
negative_price = df[df["Price"] < 0] # Creates a dataset containing only transactions with negative prices

negative_price

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Source_Sheet
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom,Year 2009-2010
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom,Year 2009-2010
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom,Year 2009-2010
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,Year 2010-2011
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,Year 2010-2011


In [28]:
zero_price = df[df["Price"] == 0] # Creates a dataset containing only transactions with a price of zero
zero_price_positive_qty = zero_price[zero_price["Quantity"] > 0]

print(f"Zero-price records: {len(zero_price):,}")
print(f"Zero-price records with positive quantity: {len(zero_price_positive_qty):,}")

Zero-price records: 6,202
Zero-price records with positive quantity: 2,745


In [29]:
zero_price_positive_qty[
    ["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID"]
].head(30)

,Invoice,StockCode,Description,Quantity,Price,Customer ID
3161,489659,21350,NaN,230,0.0,NaN
3731,489781,84292,NaN,17,0.0,NaN
4674,489825,22076,6 RIBBONS EMPIRE,12,0.0,16126.0
5904,489861,DOT,DOTCOM POSTAGE,1,0.0,NaN
6378,489882,35751C,NaN,12,0.0,NaN
6555,489898,79323G,NaN,954,0.0,NaN
6581,489903,21166,NaN,48,0.0,NaN
6781,489998,48185,DOOR MAT FAIRY CAKE,2,0.0,15658.0
7204,490015,21982,NaN,467,0.0,NaN
9249,490123,84508B,NaN,184,0.0,NaN


In [30]:
print(f"Total zero-price records: {len(zero_price):,}")
print(f"Positive quantity: {(zero_price['Quantity'] > 0).sum():,}")
print(f"Negative quantity: {(zero_price['Quantity'] < 0).sum():,}")

print(f"\nMissing Customer ID: {zero_price['Customer ID'].isna().sum():,}")
print(f"Present Customer ID: {zero_price['Customer ID'].notna().sum():,}")

print(f"\nMissing Description: {zero_price['Description'].isna().sum():,}")
print(f"Present Description: {zero_price['Description'].notna().sum():,}")

Total zero-price records: 6,202
Positive quantity: 2,745
Negative quantity: 3,457

Missing Customer ID: 6,131
Present Customer ID: 71

Missing Description: 4,382
Present Description: 1,820


#### Invoice Validation

Invoice identifies the transaction associated with each record. Invoice values are examined for different formats and prefixes because these patterns may distinguish standard sales from cancellations, adjustments, or other non-sale transaction activity.

In [31]:
invoice_prefix = df["Invoice"].astype(str).str[0].value_counts()

invoice_prefix

,count
Invoice,
5,939382
4,108489
C,19494
A,6


In [32]:
print(f"Numeric invoices: {df['Invoice'].astype(str).str.isdigit().sum():,}")
print(f"C-prefixed invoices: {df['Invoice'].astype(str).str.startswith('C').sum():,}")
print(f"A-prefixed invoices: {df['Invoice'].astype(str).str.startswith('A').sum():,}")

Numeric invoices: 1,047,871
C-prefixed invoices: 19,494
A-prefixed invoices: 6


In [33]:
c_invoices = df[df["Invoice"].astype(str).str.startswith("C")]
c_invoices["Quantity"].value_counts().sort_index(ascending=False).head(10)

,count
Quantity,
1,1
-1,8685
-2,2804
-3,1311
-4,1038
-5,382
-6,1000
-7,157
-8,328


In [34]:
c_nonnegative = c_invoices[c_invoices["Quantity"] >= 0]
c_nonnegative

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Source_Sheet
76799,C496350,M,Manual,1,2010-02-01 08:24:00,373.57,NaN,United Kingdom,Year 2009-2010


In [35]:
a_invoices = df[df["Invoice"].astype(str).str.startswith("A")]
a_invoices

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Source_Sheet
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom,Year 2009-2010
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom,Year 2009-2010
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom,Year 2009-2010
825443,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom,Year 2010-2011
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,Year 2010-2011
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,Year 2010-2011


In [36]:
print(
    f"Numeric invoices: "
    f"{df.loc[df['Invoice'].astype(str).str.isdigit(), 'Invoice'].nunique():,}"
)

print(
    f"C-prefixed invoices: "
    f"{df.loc[df['Invoice'].astype(str).str.startswith('C'), 'Invoice'].nunique():,}"
)

print(
    f"A-prefixed invoices: "
    f"{df.loc[df['Invoice'].astype(str).str.startswith('A'), 'Invoice'].nunique():,}"
)

Numeric invoices: 45,330
C-prefixed invoices: 8,292
A-prefixed invoices: 6


#### StockCode Validation

StockCode identifies the product or transaction category associated with each record. Stock codes are examined for unusual formats and non-product values because some codes may represent fees, manual entries, accounting adjustments, or other transaction activity rather than merchandise.

In [37]:
df["StockCode"].value_counts().head(20)

,count
StockCode,
85123A,5829
22423,4424
85099B,4216
21212,3318
20725,3259
84879,2960
47566,2768
21232,2747
22197,2549


In [38]:
non_numeric_stockcodes = df[
    ~df["StockCode"].astype(str).str.contains(r"\d")
]

non_numeric_stockcodes["StockCode"].value_counts()

,count
StockCode,
POST,2122
DOT,1446
M,1421
D,177
S,104
BANK CHARGES,102
ADJUST,67
AMAZONFEE,43
DCGSSGIRL,25


In [39]:
non_numeric_stockcodes[
    ["StockCode", "Description"]
].value_counts().head(30)

StockCode     Description                        
POST          POSTAGE                                2115
DOT           DOTCOM POSTAGE                         1444
M             Manual                                 1421
D             Discount                                177
S             SAMPLES                                 104
BANK CHARGES  Bank Charges                             96
AMAZONFEE     AMAZON FEE                               43
ADJUST        Adjustment by john on 26/01/2010 16      38
              Adjustment by john on 26/01/2010 17      26
DCGSSGIRL     GIRLS PARTY BAG                          23
DCGSSBOY      BOYS PARTY BAG                           21
PADS          PADS TO MATCH ALL CUSHIONS               19
CRUK          CRUK Commission                          16
BANK CHARGES   Bank Charges                             6
B             Adjust bad debt                           6
m             Manual                                    5
ADJUST        Adjustment by Peter on 24/05/2010 1       3
DCGSSBOY      update                                    1
DCGSSGIRL     update                                    1
Name: count, dtype: int64

In [40]:
special_stockcodes = [
    "POST",
    "DOT",
    "M",
    "m",
    "D",
    "S",
    "BANK CHARGES",
    "ADJUST",
    "AMAZONFEE",
    "CRUK",
    "B"
]

special_stockcode_rows = df[df["StockCode"].isin(special_stockcodes)]

print(f"Special transaction rows: {len(special_stockcode_rows):,}")
print(f"Percentage of dataset: {(len(special_stockcode_rows) / len(df) * 100):.2f}%")

Special transaction rows: 5,509
Percentage of dataset: 0.52%


#### InvoiceDate Validation

InvoiceDate represents the date and time associated with each transaction. Because the dataset combines two consecutive source periods, transaction dates are evaluated to confirm the overall date range, verify appropriate coverage within each source sheet, and identify unexpected dates that could affect time-based revenue analysis.

In [41]:
print(f"Earliest transaction: {df['InvoiceDate'].min()}")
print(f"Latest transaction: {df['InvoiceDate'].max()}")

Earliest transaction: 2009-12-01 07:45:00
Latest transaction: 2011-12-09 12:50:00


In [42]:
df.groupby("Source_Sheet")["InvoiceDate"].agg(["min", "max"])

,min,max
Source_Sheet,,
Year 2009-2010,2009-12-01 07:45:00,2010-12-09 20:01:00
Year 2010-2011,2010-12-01 08:26:00,2011-12-09 12:50:00


In [43]:
overlap = df[
    (df["InvoiceDate"] >= "2010-12-01") &
    (df["InvoiceDate"] <= "2010-12-09 23:59:59")
]

overlap.groupby("Source_Sheet").size()

,0
Source_Sheet,
Year 2009-2010,22523
Year 2010-2011,22523


In [44]:
overlap.duplicated(
    subset=[
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country"
    ],
    keep=False
).sum()

np.int64(45046)

In [45]:
transaction_fields = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country"
]

print(f"Total overlap rows: {len(overlap):,}")
print(f"Unique overlap records: {overlap[transaction_fields].drop_duplicates().shape[0]:,}")

Total overlap rows: 45,046
Unique overlap records: 22,202


In [46]:
sheet_1 = overlap[
    overlap["Source_Sheet"] == "Year 2009-2010"
][transaction_fields].drop_duplicates()

sheet_2 = overlap[
    overlap["Source_Sheet"] == "Year 2010-2011"
][transaction_fields].drop_duplicates()

print(f"Unique records in Year 2009-2010: {len(sheet_1):,}")
print(f"Unique records in Year 2010-2011: {len(sheet_2):,}")

Unique records in Year 2009-2010: 22,202
Unique records in Year 2010-2011: 22,202


In [47]:
sheet_comparison = sheet_1.merge(
    sheet_2,
    on=transaction_fields,
    how="outer",
    indicator=True
)

sheet_comparison["_merge"].value_counts()

,count
_merge,
both,22202
left_only,0
right_only,0


#### Customer ID Validation

Customer ID identifies the customer associated with each transaction. Because customer-level analysis depends on this field, missing Customer IDs are investigated to determine which transaction types are associated with unidentified customers and whether missing values occur within otherwise normal sales activity.

In [48]:
missing_customer = df[df["Customer ID"].isna()]  # Creates a dataset containing only records with a missing Customer ID

print(f"Missing Customer ID records: {len(missing_customer):,}")
print(f"Percentage of dataset: {(len(missing_customer) / len(df) * 100):.2f}%")

Missing Customer ID records: 243,007
Percentage of dataset: 22.77%


In [49]:
print(f"Positive quantities: {(missing_customer['Quantity'] > 0).sum():,}")
print(f"Negative quantities: {(missing_customer['Quantity'] < 0).sum():,}")

print(f"Positive prices: {(missing_customer['Price'] > 0).sum():,}")
print(f"Zero prices: {(missing_customer['Price'] == 0).sum():,}")
print(f"Negative prices: {(missing_customer['Price'] < 0).sum():,}")

Positive quantities: 238,801
Negative quantities: 4,206
Positive prices: 236,871
Zero prices: 6,131
Negative prices: 5


In [50]:
missing_customer_sales = missing_customer[
    (missing_customer["Quantity"] > 0) &
    (missing_customer["Price"] > 0)
]

print(f"Positive quantity and positive price: {len(missing_customer_sales):,}")
print(
    f"Percentage of missing Customer ID records: "
    f"{(len(missing_customer_sales) / len(missing_customer) * 100):.2f}%"
)

Positive quantity and positive price: 236,122
Percentage of missing Customer ID records: 97.17%


In [51]:
missing_customer_sales[
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country"
    ]
].head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom
578,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,NaN,United Kingdom
1055,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,NaN,United Kingdom
1056,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,NaN,United Kingdom
1057,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,NaN,United Kingdom
1058,489548,22195,LARGE HEART MEASURING SPOONS,1,2009-12-01 12:32:00,1.65,NaN,United Kingdom
1059,489548,22131,FOOD CONTAINER SET 3 LOVE HEART,2,2009-12-01 12:32:00,1.95,NaN,United Kingdom
1060,489548,22079,RIBBON REEL HEARTS DESIGN,10,2009-12-01 12:32:00,1.65,NaN,United Kingdom
1061,489548,22138,BAKING SET 9 PIECE RETROSPOT,3,2009-12-01 12:32:00,4.95,NaN,United Kingdom
1062,489548,22147,FELTCRAFT BUTTERFLY HEARTS,2,2009-12-01 12:32:00,1.45,NaN,United Kingdom


In [52]:
print(
    f"Distinct invoices with missing Customer ID and positive sales activity: "
    f"{missing_customer_sales['Invoice'].nunique():,}"
)

Distinct invoices with missing Customer ID and positive sales activity: 3,109


#### Description Validation

Description provides the product or transaction description associated with each StockCode. The field is evaluated for missing values and inconsistent StockCode-to-description relationships to determine whether description issues could affect product identification, classification, or downstream product-level analysis.

In [53]:
missing_description = df[df["Description"].isna()]  # Creates a dataset containing only records with a missing Description

print(f"Missing Description records: {len(missing_description):,}")
print(f"Percentage of dataset: {(len(missing_description) / len(df) * 100):.2f}%")

print(f"Positive quantities: {(missing_description['Quantity'] > 0).sum():,}")
print(f"Negative quantities: {(missing_description['Quantity'] < 0).sum():,}")

print(f"Positive prices: {(missing_description['Price'] > 0).sum():,}")
print(f"Zero prices: {(missing_description['Price'] == 0).sum():,}")
print(f"Negative prices: {(missing_description['Price'] < 0).sum():,}")

print(f"Missing Customer ID: {missing_description['Customer ID'].isna().sum():,}")

Missing Description records: 4,382
Percentage of dataset: 0.41%
Positive quantities: 1,693
Negative quantities: 2,689
Positive prices: 0
Zero prices: 4,382
Negative prices: 0
Missing Customer ID: 4,382


In [54]:
descriptions_per_stockcode = (
    df.groupby("StockCode")["Description"]
      .nunique()
      .sort_values(ascending=False)
)

print(
    f"StockCodes with more than one Description: "
    f"{(descriptions_per_stockcode > 1).sum():,}"
)

sample_stockcodes = ["20713", "21181", "23084", "22734", "22423"]

df[df["StockCode"].isin(sample_stockcodes)][
    ["StockCode", "Description"]
].drop_duplicates().sort_values(["StockCode", "Description"])

descriptions_per_stockcode.head(20)

StockCodes with more than one Description: 1,232


,Description
StockCode,
20713,9
21181,7
23084,7
22734,7
22423,7
21830,6
22719,6
47566B,6
85175,6


In [55]:
print(df["StockCode"].apply(type).value_counts())

StockCode
<class 'int'>    932385
<class 'str'>    134986
Name: count, dtype: int64


In [56]:
print(df["StockCode"].head(20).tolist())

[85048, '79323P', '79323W', 22041, 21232, 22064, 21871, 21523, 22350, 22349, 22195, 22353, '48173C', 21755, 21754, 84879, 22119, 22142, 22296, 22295]


In [57]:
df[
    df["StockCode"].astype(str).str.contains("20713", na=False)
][["StockCode", "Description"]].drop_duplicates()

,StockCode,Description
834,20713,JUMBO BAG OWLS
261489,20713,missing
318908,20713,NaN
789345,20713,wrongly marked. 23343 in box
906149,20713,wrongly coded-23343
928995,20713,found
939606,20713,Found
941043,20713,wrongly marked 23343
945852,20713,Marked as 23343
948211,20713,wrongly coded 23343


In [58]:
sample_stockcodes = ["20713", "21181", "23084", "22734", "22423"]

for stockcode in sample_stockcodes:
    print(f"\nStockCode: {stockcode}")

    descriptions = (
        df.loc[
            df["StockCode"].astype(str) == stockcode,
            "Description"
        ]
        .dropna()
        .unique()
    )

    for description in descriptions:
        print(f"  - {description}")


StockCode: 20713
  - JUMBO BAG OWLS
  - missing
  - wrongly marked. 23343 in box
  - wrongly coded-23343
  - found
  - Found
  - wrongly marked 23343
  - Marked as 23343
  - wrongly coded 23343

StockCode: 21181
  - PLEASE ONE PERSON  METAL SIGN
  - PLEASE ONE PERSON METAL SIGN
  - missing
  - on cargo order
  - adjustment
  - check
  - dotcom

StockCode: 23084
  - RABBIT NIGHT LIGHT
  - temp adjustment
  - allocate stock for dotcom orders ta
  - add stock to allocate online orders
  - for online retail orders
  - Amazon
  - website fixed

StockCode: 22734
  - SET OF 6 RIBBONS VINTAGE CHRISTMAS
  - Carton qnty was 216 not 144 as stat
  - amazon adjustment
  - amendment
  - amazon
  - amazon sales
  - FOUND

StockCode: 22423
  - REGENCY CAKESTAND 3 TIER
  - smashed
  - damaged
  - broken, uneven bottom
  - wonky bottom/broken
  - faulty
  - damages


#### Country Validation

Country identifies the geographic market associated with each transaction. The field is evaluated for missing values, inconsistent naming, unusual categories, and overall transaction distribution to determine whether geographic values can be used reliably for country-level revenue analysis.

In [59]:
country_counts = df["Country"].value_counts()

print(f"Distinct countries: {df['Country'].nunique():,}")
print(f"Missing Country records: {df['Country'].isna().sum():,}")

country_counts

Distinct countries: 43
Missing Country records: 0


,count
Country,
United Kingdom,981330
EIRE,17866
Germany,17624
France,14330
Netherlands,5140
Spain,3811
Switzerland,3189
Belgium,3123
Portugal,2620


In [60]:
uk_rows = (df["Country"] == "United Kingdom").sum()
uk_pct = uk_rows / len(df) * 100

print(f"United Kingdom records: {uk_rows:,}")
print(f"Percentage of dataset: {uk_pct:.2f}%")
print(f"Non-UK records: {len(df) - uk_rows:,}")
print(f"Non-UK percentage: {(100 - uk_pct):.2f}%")

United Kingdom records: 981,330
Percentage of dataset: 91.94%
Non-UK records: 86,041
Non-UK percentage: 8.06%


In [61]:
unspecified_rows = (df["Country"] == "Unspecified").sum()

print(f"Unspecified records: {unspecified_rows:,}")
print(f"Percentage of dataset: {(unspecified_rows / len(df) * 100):.2f}%")

Unspecified records: 756
Percentage of dataset: 0.07%


#### Cross-Field / Transaction Consistency Validation

Key transaction fields are evaluated together to identify records that do not follow the transaction patterns established during individual field validation. This analysis focuses on relationships between Invoice, Quantity, Price, Customer ID, StockCode, and Description to identify exceptions that may require separate classification or treatment during data cleaning.

In [62]:
c_invoice_exceptions = df[
    df["Invoice"].astype(str).str.startswith("C") &
    (df["Quantity"] >= 0)
]

print(f"C-prefixed invoices with non-negative Quantity: {len(c_invoice_exceptions):,}")

c_invoice_exceptions[
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "Price",
        "Customer ID",
        "Country"
    ]
]

C-prefixed invoices with non-negative Quantity: 1


,Invoice,StockCode,Description,Quantity,Price,Customer ID,Country
76799,C496350,M,Manual,1,373.57,NaN,United Kingdom


In [63]:
a_invoices = df[
    df["Invoice"].astype(str).str.startswith("A")
]

print(f"A-prefixed records: {len(a_invoices):,}")

a_invoices[
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "Price",
        "Customer ID"
    ]
].drop_duplicates()

A-prefixed records: 6


,Invoice,StockCode,Description,Quantity,Price,Customer ID
179403,A506401,B,Adjust bad debt,1,-53594.36,NaN
276274,A516228,B,Adjust bad debt,1,-44031.79,NaN
403472,A528059,B,Adjust bad debt,1,-38925.87,NaN
825443,A563185,B,Adjust bad debt,1,11062.06,NaN
825444,A563186,B,Adjust bad debt,1,-11062.06,NaN
825445,A563187,B,Adjust bad debt,1,-11062.06,NaN
